# Python API Example - Cargo Arb Breakevens

This guide is designed to provide an example of how to call the Arb Breakevens API endpoint, and store the data accordingly.

__N.B. This guide is just for Arb Breakevens data. If you're looking for other API data products (such as Netbacks, Price releases or Freight routes), please refer to their according code example files.__ 

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html


In [ ]:
# Input the path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## Netbacks Reference Data

Calling the netbacks reference data endpoint to retrieve available FOB Ports and their corresponding UUIDs

In [ ]:
# Define the function for listing all netback reference data
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    # retrieve release dates separately
    reldates = content["data"]["staticData"]["sparkReleases"]

    # return reference data
    refdata_dict = content["data"]

    return reldates, refdata_dict

# Fetch reference data:
reldates, refdata_dict = list_netbacks(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['staticData']['fobPorts'])
available_df.head(3)

## Defining Function to Fetch Arb Breakevens data

The endpoint is divided into 2 sub-endpoints: "/freight" and "/jkm-ttf". Here, the function treats the breakeven type as an input to choose the required sub-endpoint

More information on parameters for this endpoint can be found on the API Website:

https://www.sparkcommodities.com/api/lng-cargo/netbacks-arb-breakevens.html

In [ ]:


def fetch_breakevens(access_token, fob_port_uuid, via_point=None, breakeven='jkm-ttf', start=None, end=None, format='json'):
    
    #For a route, fetch then display the route details
    #https://api.sparkcommodities.com/v1.0/netbacks/arb-breakevens/
    


    query_params = breakeven + '/' + "?fob-port={}".format(fob_port_uuid)

    if via_point is not None:
        query_params += "&via-point={}".format(via_point)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)

    uri = "/v1.0/netbacks/arb-breakevens/{}".format(query_params)
    print(uri)
    content = do_api_get_query(
        uri="/v1.0/netbacks/arb-breakevens/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        my_dict = content['data']
    else:
        my_dict = content.decode('utf-8')
        my_dict = pd.read_csv(StringIO(my_dict))

    return my_dict




### Choosing FOB Port

In [ ]:
# Select Route
tsab = available_df[available_df["name"] == 'Sabine Pass']["uuid"].iloc[0]

### JSON Response Example

In [ ]:
## Calling that function and storing the output - JSON version
json_data = fetch_breakevens(access_token, fob_port_uuid=tsab, via_point='cogh', breakeven='jkm-ttf', start='2025-07-20', end='2025-07-30')
json_data

## CSV example

In [ ]:
# calling the same data in CSV format. This option automatically converts the data to a pandas dataframe.
csv_df = fetch_breakevens(access_token, fob_port_uuid=tsab, via_point='cogh', breakeven='jkm-ttf', start='2025-07-20', end='2025-07-30', format='csv')
csv_df